## Topic 5: Qdrant's Data Model — Collections, Points & Payloads

### Why Do We Need a Data Model?

After generating embeddings, we need a structured way to store them.

A vector database cannot simply store vectors because a vector alone has no meaning.

For every document chunk, we also need to store:

- The original text.
- Where the text came from.
- Document type.
- Page number.
- Chunk number.
- Any additional metadata.

Qdrant organizes all this information using three core concepts:

- **Collection**
- **Point**
- **Payload**

Understanding these three concepts makes every Qdrant operation—insert, search, filter, retrieve, and update—easy to understand.

---

## Collection

### Definition

A **Collection** is a container that stores related vectors.

It is the highest-level object inside Qdrant.

Every collection contains multiple points.

A collection is similar to a SQL table because it groups similar records together.

Unlike a SQL table, every point stores an embedding along with its metadata.

---

### Example

Suppose we build an Email FAQ chatbot.

We create one collection named:

```
fd_knowledge_base
```

This collection stores:

- FD Policies
- Product Guides
- FAQs

Every document chunk belonging to the knowledge base is stored inside this collection.

---

### Characteristics

A collection defines:

- Vector dimension (384 in this project)
- Distance metric (Cosine Similarity)
- Index configuration (HNSW)

All vectors inside the collection must have the same dimension.

---

### Advantages

- Organizes related vectors together.
- Provides a single search endpoint.
- Supports filtering across all document types.
- Simplifies application design.

---

### Disadvantages

- Vector dimension cannot be changed after creation.
- Changing the embedding model usually requires recreating the collection.

---

## Point

### Definition

A **Point** is one record inside a collection.

It is the smallest unit stored in Qdrant.

Every Point contains three components:

- Unique ID
- Vector (Embedding)
- Payload (Metadata)

Without a Point, Qdrant has nothing to search.

---

### Example

One document chunk becomes one Point.

```
Point

ID

3267135047

Vector

[384 floating-point numbers]

Payload

{
    text,
    source,
    doc_type,
    page,
    chunk_index
}
```

---

### Advantages

- Self-contained storage.
- Supports semantic search.
- Stores metadata together with embeddings.
- Easy to retrieve and update.

---

### Disadvantages

- Every point requires a unique ID.
- Duplicate IDs replace existing points.

---

## Payload

### Definition

A **Payload** is the metadata attached to a Point.

It stores information about the vector but is **not** used to calculate semantic similarity.

Payload enables filtering, retrieval, and citations.

---

### Example

```json
{
    "text": "...",
    "source": "FD_Policy.json",
    "doc_type": "policy",
    "page": 1,
    "chunk_index": 0
}
```

During retrieval:

- Vector finds similar documents.
- Payload explains what those documents are.

---

### Advantages

- Stores useful metadata.
- Enables metadata filtering.
- Makes search results self-contained.
- Supports document citations.

---

### Disadvantages

- Schema is not enforced.
- Inconsistent field names can break filtering.
- Large payloads increase storage size.

---

## Point ID

### Definition

Every Point must have a unique identifier.

Qdrant uses this ID to identify, retrieve, update, and delete points.

IDs can be:

- Integer
- UUID

This project generates deterministic integer IDs.

---

### Why Deterministic IDs?

Suppose a document is ingested today.

Tomorrow the same document is updated.

If the same ID is generated:

- Old point is updated.
- No duplicate is created.

If a random ID is generated:

- Old point remains.
- New point is inserted.
- Duplicate vectors appear.

---

### Advantages

- Supports idempotent upserts.
- Prevents duplicate points.
- Simplifies incremental ingestion.

---

### Disadvantages

- Requires deterministic ID generation.
- Hash collisions are theoretically possible (very rare).

---

## Upsert

### Definition

Upsert means:

- **Insert** if the Point ID does not exist.
- **Update** if the Point ID already exists.

This is the primary operation used to store vectors inside Qdrant.

---

### Example

First ingestion:

```
ID = 3267135047

Text

Premature withdrawal incurs a penalty.
```

Second ingestion with the same ID:

```
ID = 3267135047

Text

UPDATED: Premature withdrawal penalty is 1 percent.
```

Instead of creating another point, Qdrant replaces the existing one.

---

### Advantages

- Prevents duplicate points.
- Ideal for incremental ingestion.
- Simple API.
- Automatic insert/update handling.

---

### Disadvantages

- Replaces the entire point.
- Missing payload fields are removed unless included again.

---

## Payload Index

### Definition

Payload fields are stored inside every Point.

Searching using these fields becomes faster when a Payload Index is created.

Without an index, Qdrant scans every payload during filtering.

With an index, Qdrant directly locates matching points.

---

### Example

Suppose the collection contains one million points.

Filter:

```
doc_type = "faq"
```

Without Payload Index

- Every payload is checked.

With Payload Index

- Qdrant directly finds FAQ points.

---

### Advantages

- Faster filtering.
- Better scalability.
- Improved query performance.

---

### Disadvantages

- Uses additional storage.
- Slightly increases ingestion time because indexes must also be updated.

---

## Retrieve

### Definition

Retrieve fetches one or more Points using their IDs.

Unlike semantic search, no query embedding is required.

---

### Example

Retrieve:

```
ID = 3267135047
```

Qdrant immediately returns:

- Payload
- Vector (optional)
- Metadata

No similarity search is performed.

---

### Advantages

- Extremely fast.
- No embedding generation.
- Useful for direct lookups.

---

### Disadvantages

- ID must already be known.
- Cannot perform semantic search.

---

## Scroll

### Definition

Scroll iterates through Points stored inside a collection.

Unlike semantic search, Scroll does not require a query vector.

It is mainly used for administrative operations.

---

### Example

Scroll all Policy documents.

Scroll all FAQ documents.

Scroll all documents from a specific source.

---

### Advantages

- Useful for auditing.
- Supports bulk updates.
- Supports bulk deletion.
- Useful for re-embedding pipelines.

---

### Disadvantages

- Not intended for semantic retrieval.
- Slower than searching by ID.

---

## Relationship Between Collection, Point and Payload

A Collection contains multiple Points.

Each Point contains:

- Unique ID
- Embedding Vector
- Payload

The Payload stores metadata such as:

- Original text
- Source file
- Document type
- Page number
- Chunk number

During semantic search:

- Vector is used to calculate similarity.
- Payload is used for filtering and displaying results.

---

## Real-World Issues, Edge Cases & Debugging

### Collection

- Vector dimension cannot be changed after creation.
- Changing the embedding model usually requires recreating the collection.

---

### Point

- Duplicate IDs overwrite existing Points.
- Random IDs create duplicate documents during re-ingestion.

---

### Payload

- Inconsistent field names cause filtering failures.
- Large payloads increase storage requirements.

---

### Payload Index

- Filtering large collections without an index becomes slow.
- Every frequently filtered field should have an index in production.

---

### Upsert

- Upsert replaces the complete Point.
- Partial updates require sending the complete payload again.

---

## Common Mistakes & Production Failures

- Using random IDs during every ingestion.
- Forgetting to store original text inside the Payload.
- Assuming Qdrant enforces payload schema.
- Forgetting to create Payload Indexes for frequently filtered fields.
- Changing embedding models without recreating the collection.

---

## Lead-Level Interview Questions

### Q1. Why does every Point require a unique ID?

**Answer**

The ID uniquely identifies every Point inside a collection. It enables retrieval, updates, deletion, and idempotent upserts. Without unique IDs, duplicate Points would accumulate during repeated ingestion.

---

### Q2. Why store the original text inside the Payload when embeddings already exist?

**Answer**

Embeddings are numerical vectors and cannot be directly displayed to users. Storing the original text inside the Payload allows search results to be immediately returned without performing another lookup in the original documents.

---

### Q3. Why should production systems use deterministic IDs?

**Answer**

Deterministic IDs ensure that re-ingesting the same document updates the existing Point instead of creating duplicate vectors. This makes ingestion idempotent and prevents stale search results.

---

### Q4. Why create a Payload Index?

**Answer**

Without a Payload Index, Qdrant scans every Point while applying filters. Payload indexes allow Qdrant to locate matching Points much faster, significantly improving filtering performance on large collections.

---

### Q5. What is the difference between Search, Retrieve, and Scroll?

**Answer**

- **Search** finds similar vectors using semantic similarity.
- **Retrieve** fetches Points directly using their IDs.
- **Scroll** iterates through stored Points without performing semantic search.

Each operation serves a different purpose and should be chosen based on the application's requirements.

---

## Key Takeaways

- A **Collection** is a container that stores related Points.
- A **Point** consists of an ID, Vector, and Payload.
- A **Payload** stores metadata used for filtering and retrieval.
- **Upsert** inserts new Points or updates existing ones.
- **Deterministic IDs** prevent duplicate Points during re-ingestion.
- **Payload Indexes** improve filtering performance.
- **Retrieve** fetches by ID, while **Scroll** iterates through stored Points without semantic search.
```


In [1]:
"""
qdrant_collections_points.py
------------------------------
Demonstrates Qdrant's core data model: collections, points, payloads.

Shows:
  - Creating a collection
  - Generating deterministic point IDs (idempotent upserts)
  - Upserting points with payload mirroring the Document shape
  - Adding a payload index for filtered search performance
  - Retrieving a specific point by ID
  - Scrolling through all points without a query vector
  - Demonstrating idempotent upsert (re-upsert updates, not duplicates)

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,          # cosine, dot, euclidean -- how to measure vector closeness
    VectorParams,      # vector size + distance metric config for a collection
    PointStruct,       # one point: id + vector + payload
    PayloadSchemaType, # tells Qdrant what type of payload index to create
    Filter,            # wraps filter conditions for a search or scroll
    FieldCondition,    # one condition on a payload field
    MatchValue,        # "must equal this exact value" match type
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384        # must match the embedding model's output size
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# in-memory client -- no Docker, no server needed
client = QdrantClient(":memory:")

# load embedding model once -- reused for every encode() call
model = SentenceTransformer(MODEL_NAME)

# sample documents in the same shape the ingestion chapter produces
DOCUMENTS = [
    {
        "page_content": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
        "metadata": {"source": "FD_Policy.json", "doc_type": "policy", "page": 1, "chunk_index": 0},
    },
    {
        "page_content": "This penalty does not apply if the FD is closed due to the death of the depositor.",
        "metadata": {"source": "FD_Policy.json", "doc_type": "policy", "page": 1, "chunk_index": 1},
    },
    {
        "page_content": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
        "metadata": {"source": "FD_Product_Guide.json", "doc_type": "product", "page": 2, "chunk_index": 0},
    },
    {
        "page_content": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.",
        "metadata": {"source": "FD_FAQ.json", "doc_type": "faq", "page": 1, "chunk_index": 0},
    },
    {
        "page_content": "Can I withdraw my FD before maturity? Yes, subject to a penalty.",
        "metadata": {"source": "FD_FAQ.json", "doc_type": "faq", "page": 1, "chunk_index": 1},
    },
]


def make_point_id(source: str, chunk_index: int) -> int:
    """Deterministic integer ID from source path + chunk index.
    Same inputs always produce the same ID so re-upserting the same
    chunk updates it in place instead of creating a duplicate point."""
    key = f"{source}::{chunk_index}"
    # take first 8 hex chars of md5 and convert to int -- small enough for Qdrant's uint64 ID
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


def create_collection() -> None:
    # check existing collections to avoid crashing on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    # delete the old collection if present -- clean slate for this demo
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,          # every point's vector must be exactly this many floats
            distance=Distance.COSINE,  # cosine similarity -- correct for normalized vectors
        ),
    )
    print(f"Created collection '{COLLECTION_NAME}'")


def add_payload_index() -> None:
    """Adds a keyword payload index on 'doc_type'.
    Without this, a filter on doc_type scans every payload record.
    With this, Qdrant resolves the filter via the index -- much faster at scale."""
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="doc_type",                  # the payload field to index
        field_schema=PayloadSchemaType.KEYWORD,  # keyword index = exact string match
    )
    print("Added payload index on 'doc_type'")


def upsert_documents(documents: list) -> None:
    # extract text strings for batch embedding
    texts = [d["page_content"] for d in documents]

    # embed all at once -- normalize so dot product == cosine similarity
    embeddings = model.encode(texts, normalize_embeddings=True)

    points = []
    for i, doc in enumerate(documents):
        meta = doc["metadata"]

        # deterministic ID -- same source + chunk_index always gives the same ID
        point_id = make_point_id(meta["source"], meta["chunk_index"])

        points.append(
            PointStruct(
                id=point_id,
                vector=embeddings[i].tolist(),  # numpy array -> plain list for serialization
                payload={
                    "text": doc["page_content"],    # store raw text for self-contained retrieval
                    "source": meta["source"],        # which file this chunk came from
                    "doc_type": meta["doc_type"],    # used for payload filtering
                    "page": meta["page"],            # page within the source document
                    "chunk_index": meta["chunk_index"],  # position within the page
                },
            )
        )

    # upsert = insert if new ID, replace entirely if ID already exists
    client.upsert(collection_name=COLLECTION_NAME, points=points)

    info = client.get_collection(COLLECTION_NAME)
    print(f"Upserted {len(points)} points | Total in collection: {info.points_count}")


def retrieve_by_id(source: str, chunk_index: int) -> None:
    """Fetches one specific point by its deterministic ID -- no query vector needed."""
    point_id = make_point_id(source, chunk_index)

    results = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=[point_id],
        with_payload=True,   # include the payload fields in the result
        with_vectors=False,  # skip the raw embedding -- not needed for inspection
    )

    if results:
        print(f"\nRetrieved point ID={point_id}:")
        print(f"  text     : {results[0].payload['text'][:80]}")
        print(f"  doc_type : {results[0].payload['doc_type']}")
        print(f"  source   : {results[0].payload['source']}")
    else:
        print(f"No point found for ID={point_id}")


def scroll_all(doc_type_filter: str = None) -> None:
    """Iterates through points without a query vector.
    Used for auditing, bulk re-embedding, or deletion by filter --
    not for similarity search."""

    # build a filter if a doc_type was given, otherwise scroll everything
    scroll_filter = None
    if doc_type_filter:
        scroll_filter = Filter(
            must=[
                FieldCondition(
                    key="doc_type",
                    match=MatchValue(value=doc_type_filter),
                )
            ]
        )

    # scroll returns (results, next_page_offset) -- we only need the results here
    results, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=scroll_filter,
        with_payload=True,
        with_vectors=False,
        limit=100,  # max points to return in one scroll call
    )

    label = f"doc_type='{doc_type_filter}'" if doc_type_filter else "all"
    print(f"\nScroll ({label}): {len(results)} points")
    for r in results:
        print(f"  [{r.payload['doc_type']}] {r.payload['text'][:60]}")


def demonstrate_idempotent_upsert() -> None:
    """Re-upserts a point that already exists (same deterministic ID).
    Point count stays the same -- it updated in place, not duplicated."""

    # record the count before re-upserting
    count_before = client.get_collection(COLLECTION_NAME).points_count

    # same source + chunk_index as the first document -- same ID will be generated
    updated_doc = {
        "page_content": "UPDATED: Premature withdrawal penalty is 1 percent.",
        "metadata": {
            "source": "FD_Policy.json",
            "doc_type": "policy",
            "page": 1,
            "chunk_index": 0,  # same chunk_index -> same ID -> in-place update
        },
    }
    upsert_documents([updated_doc])

    count_after = client.get_collection(COLLECTION_NAME).points_count

    print(f"\nIdempotent upsert: before={count_before}, after={count_after}")
    print("  Count unchanged -- updated in place, no duplicate created")

    # confirm the content was actually updated
    retrieve_by_id("FD_Policy.json", chunk_index=0)


# ----------------------------------------------------------------------
# Run all demos in order
# ----------------------------------------------------------------------

# step 1: create collection
create_collection()

# step 2: add payload index BEFORE upserting so Qdrant indexes as points arrive
add_payload_index()

# step 3: embed and upsert all sample documents
upsert_documents(DOCUMENTS)

# step 4: fetch one specific point by its deterministic ID
retrieve_by_id("FD_Policy.json", chunk_index=0)

# step 5: scroll through all points (no query vector)
scroll_all()

# step 6: scroll only FAQ points using the payload index
scroll_all(doc_type_filter="faq")

# step 7: re-upsert one point and show the count stays the same
demonstrate_idempotent_upsert()


Created collection 'fd_knowledge_base'
Added payload index on 'doc_type'
Upserted 5 points | Total in collection: 5

Retrieved point ID=3267135047:
  text     : Premature withdrawal incurs a 1 percent penalty on the applicable rate.
  doc_type : policy
  source   : FD_Policy.json

Scroll (all): 5 points
  [faq] What is the minimum amount to open an FD? The minimum deposi
  [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  [product] Senior citizens receive an additional 0.5 percent interest o
  [policy] This penalty does not apply if the FD is closed due to the d
  [policy] Premature withdrawal incurs a 1 percent penalty on the appli

Scroll (doc_type='faq'): 2 points
  [faq] What is the minimum amount to open an FD? The minimum deposi
  [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
Upserted 1 points | Total in collection: 5

Idempotent upsert: before=5, after=5
  Count unchanged -- updated in place, no duplicate created

Retrieved point ID=32671

C:\Users\pauls\AppData\Local\Temp\ipykernel_42232\3686356635.py:99: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(
